<a href="https://colab.research.google.com/github/Datadog-995/Cleaned-Butcher-Sales-Portfolio/blob/main/7_financial_transactions_final_cleanedi_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

# 1. Load the dirty dataset
# Replace with your actual local file path if different
# The original file 'dirty_financial_transactions.csv' was not found.
# Using 'dirty_financial_transactions 2.csv' which is available in the environment.
df = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Basic Shape & Data Type Audit
print("--- DATASET SHAPE ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}\n")

print("--- DATA TYPE INITIAL AUDIT ---")
print(df.dtypes)
print("\n--- MISSING VALUE MAP ---")
print(df.isnull().sum())

--- DATASET SHAPE ---
Total Rows: 100000, Total Columns: 8

--- DATA TYPE INITIAL AUDIT ---
Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Product_Name           object
Quantity              float64
Price                  object
Payment_Method         object
Transaction_Status     object
dtype: object

--- MISSING VALUE MAP ---
Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64


## 3. Data Cleaning and Preprocessing

In [6]:
# Convert 'Transaction_Date' to datetime
# Using errors='coerce' will turn unparseable dates into NaT (Not a Time)
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Convert 'Price' to numeric, handling errors
# 'errors='coerce' will turn non-numeric values into NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Convert 'Quantity' to numeric (it's already float, but good for consistency)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

print("--- DATA TYPES AFTER CONVERSION ---")
print(df.dtypes)

print("\n--- MISSING VALUES AFTER CONVERSION ---")
print(df.isnull().sum())

--- DATA TYPES AFTER CONVERSION ---
Transaction_ID                object
Transaction_Date      datetime64[ns]
Customer_ID                   object
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object

--- MISSING VALUES AFTER CONVERSION ---
Transaction_ID         5018
Transaction_Date      68261
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 40137
Payment_Method            0
Transaction_Status    16679
dtype: int64


### Handle Missing Values

In [7]:
# The user wants to keep all data and not remove any missing values.
# Instead of dropping, we will fill missing values with appropriate placeholders.

# Fill missing 'Transaction_ID' and 'Customer_ID' with 'Unknown'
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_ID'] = df['Transaction_ID'].fillna('Unknown')
df['Customer_ID'] = df['Customer_ID'].fillna('Unknown')

# Fill missing 'Transaction_Date' (NaT values) with a placeholder date (e.g., 1900-01-01)
# This keeps the column as datetime, allowing for date-based operations later if needed.
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_Date'] = df['Transaction_Date'].fillna(pd.Timestamp('1900-01-01'))

# Fill missing 'Quantity' and 'Price' with 0, assuming a missing value means no quantity/price
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Quantity'] = df['Quantity'].fillna(0)
df['Price'] = df['Price'].fillna(0)

print("\n--- MISSING VALUES AFTER FILLING (NO DROPS) ---")
print(df.isnull().sum())


--- MISSING VALUES AFTER FILLING (NO DROPS) ---
Transaction_ID            0
Transaction_Date          0
Customer_ID               0
Product_Name              0
Quantity                  0
Price                     0
Payment_Method            0
Transaction_Status    16679
dtype: int64


### Clean and Standardize `Transaction_Status`

In [8]:
# Standardize 'Transaction_Status' to a consistent format
df['Transaction_Status'] = df['Transaction_Status'].str.strip().str.capitalize()

# Fill missing 'Transaction_Status' with 'Unknown'
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_Status'] = df['Transaction_Status'].fillna('Unknown')

print("\n--- UNIQUE TRANSACTION STATUS VALUES ---")
print(df['Transaction_Status'].unique())

print("\n--- MISSING VALUES AFTER STATUS CLEANING ---")
print(df.isnull().sum())


--- UNIQUE TRANSACTION STATUS VALUES ---
['Unknown' 'Pending' 'Completed' 'Complete' 'Failed']

--- MISSING VALUES AFTER STATUS CLEANING ---
Transaction_ID        0
Transaction_Date      0
Customer_ID           0
Product_Name          0
Quantity              0
Price                 0
Payment_Method        0
Transaction_Status    0
dtype: int64


### Address Inconsistencies: Negative Quantities

In [9]:
# Identify and handle negative quantities
negative_quantities = df[df['Quantity'] < 0]
print(f"Number of transactions with negative quantities: {len(negative_quantities)}")

# Option 1: Convert negative quantities to their absolute values (assuming they represent returns or cancellations with positive magnitude)
# df['Quantity'] = df['Quantity'].abs()

# Option 2: Drop rows with negative quantities if they are considered invalid data entries.
# Given the nature of transactions, a negative quantity might imply a return or correction.
# For simplicity, let's convert them to positive values, but this depends on business logic.
df['Quantity'] = df['Quantity'].abs()

print("\n--- DATASET SHAPE AFTER CLEANING ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

print("\n--- FINAL DATA TYPES ---")
print(df.dtypes)

print("\n--- FINAL MISSING VALUE MAP ---")
print(df.isnull().sum())

Number of transactions with negative quantities: 31619

--- DATASET SHAPE AFTER CLEANING ---
Total Rows: 100000, Total Columns: 8

--- FINAL DATA TYPES ---
Transaction_ID                object
Transaction_Date      datetime64[ns]
Customer_ID                   object
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object

--- FINAL MISSING VALUE MAP ---
Transaction_ID        0
Transaction_Date      0
Customer_ID           0
Product_Name          0
Quantity              0
Price                 0
Payment_Method        0
Transaction_Status    0
dtype: int64


### Display cleaned data sample

In [10]:
display(df.head())

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,5.0,0.000000,pay pal,Unknown
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.342025,creditcard,Pending
2,T0003,1900-01-01,C2919,Tablet,4.0,810.993012,credit card,Completed
3,T0004,2020-08-17,C3009,Tab,7.0,868.608341,PayPal,Pending
4,T0005,1900-01-01,C3488,Coffee Machine,10.0,-763.122449,PayPal,Completed


## 4. Save Cleaned Dataset to Google Drive

In [11]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the path in Google Drive where you want to save the file
# You can change 'My Drive/Colab Notebooks/' to your preferred folder structure.
output_path = '/content/drive/My Drive/cleaned_financial_transactions.csv'

# Save the DataFrame to a CSV file in Google Drive
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
df.to_csv(output_path, index=False)

print(f"Dataset successfully saved to: {output_path}")

Dataset successfully saved to: /content/drive/My Drive/cleaned_financial_transactions.csv
